In [3]:
# Setup -- we'll run SQL right inside this notebook using sqlite3
import sqlite3
import pandas as pd

# Create an in-memory database (wipes when the kernel restarts -- perfect for teaching)
conn = sqlite3.connect(':memory:')

# Helper: run a query and return a clean DataFrame
def q(sql):
    return pd.read_sql_query(sql, conn)

# Helper: run DDL / DML statements (CREATE, INSERT, UPDATE, DELETE)
def run(sql):
    conn.executescript(sql)
    conn.commit()

print('SQL engine ready.')
print('Use q("SELECT ...") to query, run("CREATE / INSERT / ...") for everything else.')

SQL engine ready.
Use q("SELECT ...") to query, run("CREATE / INSERT / ...") for everything else.


In [4]:
# Optional -- point the notebook at olympics.db (if it sits next to this notebook)
import os
if os.path.exists('olympics.db'):
    conn = sqlite3.connect('olympics.db')
    def q(sql): return pd.read_sql_query(sql, conn)
    def run(sql): conn.executescript(sql); conn.commit()
    print('Now querying the real Olympics database.')
    print(q('SELECT COUNT(*) AS n_athletes FROM person'))
else:
    print('olympics.db not found -- use sqliteonline for the exercises.')

Now querying the real Olympics database.
   n_athletes
0      128854


In [47]:
# Task 1: Find the average age of competitors who have won at least one medal, grouped by the type of medal they won. Use a correlated subquery to achieve this.

q('''
SELECT
    m.medal_name,
    AVG(gc.age) AS avg_age
FROM competitor_event ce
JOIN medal m
    ON ce.medal_id = m.id
JOIN games_competitor gc
    ON ce.competitor_id = gc.id
JOIN person p
    ON gc.person_id = p.id
WHERE m.medal_name != 'NA'
GROUP BY m.medal_name;
''')

,medal_name,avg_age
0,Bronze,25.881749
1,Gold,25.915993
2,Silver,26.006659


In [30]:
# Task 2: Identify the top 5 regions with the highest number of unique competitors who have participated in more than 3 different events. Use nested subqueries to filter and aggregate the data.
q('''
SELECT
    nr.region_name,
    COUNT(DISTINCT pr.person_id) AS unique_competitors
FROM person_region pr
JOIN noc_region nr
    ON pr.region_id = nr.id
WHERE pr.person_id IN (
    SELECT gc.person_id
    FROM games_competitor gc
    WHERE gc.id IN (
        SELECT ce.competitor_id
        FROM competitor_event ce
        GROUP BY ce.competitor_id
        HAVING COUNT(DISTINCT ce.event_id) > 3
    )
)
GROUP BY nr.region_name
ORDER BY unique_competitors DESC
LIMIT 5;
''')

,region_name,unique_competitors
0,USA,444
1,France,271
2,Canada,243
3,Japan,220
4,Germany,211


In [36]:
#  Create a temporary table to store the total number of medals won by each competitor and filter to show only those who have won more than 2 medals. Use subqueries to aggregate the data.

conn.execute('''
CREATE TEMP TABLE competitor_medal_counts11 AS
SELECT *
FROM (
    SELECT
        ce.competitor_id,
        COUNT(*) AS total_medals
    FROM competitor_event ce
    JOIN medal m
        ON ce.medal_id = m.id
    WHERE m.medal_name != 'NA'
    GROUP BY ce.competitor_id
) AS medal_counts
WHERE total_medals > 2;
''')

In [37]:
q('''
SELECT *
FROM competitor_medal_counts11;
''')

,competitor_id,total_medals
0,24,4
1,29,3
2,575,3
3,1294,3
4,1295,4
...,...,...
760,179508,3
761,179818,3
762,179819,3
763,179980,3


In [40]:
conn.execute('''
DROP TABLE IF EXISTS temp_competitor_analysis;
''')

conn.execute('''
CREATE TEMP TABLE temp_competitor_analysis AS
SELECT DISTINCT
    ce.competitor_id
FROM competitor_event ce;
''')

In [41]:
conn.execute('''
DELETE FROM temp_competitor_analysis
WHERE competitor_id NOT IN (
    SELECT DISTINCT ce.competitor_id
    FROM competitor_event ce
    JOIN medal m
        ON ce.medal_id = m.id
    WHERE m.medal_name != 'NA'
);
''')

In [42]:
q('''
SELECT *
FROM temp_competitor_analysis;
''')

,competitor_id
0,4
1,21
2,23
3,24
4,25
...,...
34310,180223
34311,180225
34312,180226
34313,180240


In [43]:
# Task 1: Update the heights of competitors based on the average height of competitors from the same region. Use a correlated subquery within the UPDATE statement.
conn.execute('''
DROP TABLE IF EXISTS temp_person_height;
''')

conn.execute('''
CREATE TEMP TABLE temp_person_height AS
SELECT
    p.id AS person_id,
    p.full_name,
    p.height,
    pr.region_id
FROM person p
JOIN person_region pr
    ON p.id = pr.person_id;
''')

In [44]:
conn.execute('''
UPDATE temp_person_height AS tph
SET height = (
    SELECT AVG(tph2.height)
    FROM temp_person_height AS tph2
    WHERE tph2.region_id = tph.region_id
      AND tph2.height IS NOT NULL
)
WHERE tph.height IS NULL;
''')

In [45]:
q('''
SELECT *
FROM temp_person_height
LIMIT 10;
''')

,person_id,full_name,height,region_id
0,1,A Dijiang,180,42
1,2,A Lamusi,170,42
2,3,Gunnar Nielsen Aaby,0,56
3,4,Edgar Lindenau Aabye,0,56
4,5,Christine Jacoba Aaftink,185,146
5,6,Per Knut Aaland,188,217
6,7,John Aalberg,183,217
7,8,Cornelia 'Cor'' Aalten (-Strannood)',168,146
8,9,Antti Sami Aalto,186,69
9,10,Einar Ferdinand 'Einari'' Aalto',0,69


In [46]:
# Task 2: Insert new records into a temporary table for competitors who participated in more than one event in the same games and list their total number of events participated. Use nested subqueries for filtering.
conn.execute('''
DROP TABLE IF EXISTS temp_multi_event_competitors;
''')

conn.execute('''
CREATE TEMP TABLE temp_multi_event_competitors (
    competitor_id INTEGER,
    games_id INTEGER,
    total_events INTEGER
);
''')

In [47]:
conn.execute('''
INSERT INTO temp_multi_event_competitors
SELECT
    competitor_id,
    games_id,
    total_events
FROM (
    SELECT
        gc.id AS competitor_id,
        gc.games_id,
        COUNT(DISTINCT ce.event_id) AS total_events
    FROM games_competitor gc
    JOIN competitor_event ce
        ON gc.id = ce.competitor_id
    GROUP BY gc.id, gc.games_id
) AS event_counts
WHERE total_events > 1;
''')

In [48]:
q('''
SELECT *
FROM temp_multi_event_competitors
LIMIT 20;
''')

,competitor_id,games_id,total_events
0,5,5,2
1,6,6,2
2,7,7,2
3,8,6,4
4,9,7,4
5,10,6,4
6,11,7,4
7,12,8,2
8,20,14,2
9,21,3,2


In [49]:
# Task 3: Identify regions where the average number of medals won per competitor is greater than the overall average. Use subqueries to calculate and compare averages.

q('''
SELECT
    region_medals.region_id,
    AVG(region_medals.total_medals) AS avg_medals_per_competitor
FROM (
    SELECT
        pr.region_id,
        gc.person_id,
        COUNT(ce.medal_id) AS total_medals
    FROM games_competitor gc
    JOIN competitor_event ce
        ON gc.id = ce.competitor_id
    JOIN medal m
        ON ce.medal_id = m.id
    JOIN person_region pr
        ON gc.person_id = pr.person_id
    WHERE m.medal_name != 'NA'
    GROUP BY pr.region_id, gc.person_id
) AS region_medals
GROUP BY region_medals.region_id
HAVING AVG(region_medals.total_medals) > (
    SELECT AVG(overall_medals.total_medals)
    FROM (
        SELECT
            gc.person_id,
            COUNT(ce.medal_id) AS total_medals
        FROM games_competitor gc
        JOIN competitor_event ce
            ON gc.id = ce.competitor_id
        JOIN medal m
            ON ce.medal_id = m.id
        WHERE m.medal_name != 'NA'
        GROUP BY gc.person_id
    ) AS overall_medals
);
''')

,region_id,avg_medals_per_competitor
0,1,2.000000
1,3,1.500000
2,13,1.542824
3,14,1.431250
4,16,1.600000
5,26,1.483333
6,27,1.428571
7,42,1.487988
8,50,2.000000
9,55,1.446281


In [50]:
# Task 4: Create a temporary table to track competitors’ participation across different seasons and identify those who have participated in both Summer and Winter games.

conn.execute('''
DROP TABLE IF EXISTS temp_competitor_seasons;
''')

conn.execute('''
CREATE TEMP TABLE temp_competitor_seasons AS
SELECT
    gc.person_id,
    p.full_name,
    COUNT(DISTINCT g.season) AS number_of_seasons,
    GROUP_CONCAT(DISTINCT g.season) AS seasons_participated
FROM games_competitor gc
JOIN games g
    ON gc.games_id = g.id
JOIN person p
    ON gc.person_id = p.id
GROUP BY gc.person_id, p.full_name
HAVING COUNT(DISTINCT g.season) = 2;
''')


q('''
SELECT *
FROM temp_competitor_seasons;
''')

,person_id,full_name,number_of_seasons,seasons_participated
0,770,Marcus Adam,2,"Summer,Winter"
1,4660,Shinji Aoto,2,"Summer,Winter"
2,5429,Karl Wilhelm Konrad Arwe (Andersson-),2,"Summer,Winter"
3,5605,Jorun Askersrud-Tangen,2,"Summer,Winter"
4,7978,Bryan Barnett,2,"Summer,Winter"
...,...,...,...,...
153,129491,Theresa Weld-Blanchard (-Barnes),2,"Summer,Winter"
154,130148,Hayley Marie Wickenheiser,2,"Winter,Summer"
155,130626,Lauryn Chenet Williams,2,"Summer,Winter"
156,131183,"Christine Diane ""Chris"""" Witty""",2,"Winter,Summer"
